# Data exploration and audit

Audit of the Geneva building-permit export before any cleaning, splitting or modeling decision is made.

In [1]:
import sys

sys.path.insert(0, "..")

## Basic inspection
Load the export and see what is in it - shape, columns, types, first rows.

In [2]:
# Loading the data in the Pandas DataFrame - outputing the shape of the data.
from src.data import load_source

df = load_source()
df.shape

(197224, 16)

In [3]:
df.head()

,OBJECTID,ID_DOSSIER,TYPE_DOSSIER,TYPE_OPERATION,NOM_DOSSIER,NO_DOSSIER,NO_COMPLEMENTAIRE,STATUT,DATE_DEPOT,DESCRIPTION,OPERATION,ORIGINE,DATE_MAJ_2,LIEN_SAD,E,N
0,45,DD 17497/1,DD,--,Demande définitive,17497,1,-,NaT,non renseigné,Non renseigné (valeur par défaut à l'importati...,ANC,2002-02-26,<NA>,2498512.96,1125146.27
1,46,DD 87127/1,DD,--,Demande définitive,87127,1,-,NaT,non renseigné,Non renseigné (valeur par défaut à l'importati...,ANC,2002-02-26,<NA>,2498522.16,1125164.87
2,47,DD 18791/1,DD,--,Demande définitive,18791,1,-,NaT,non renseigné,Non renseigné (valeur par défaut à l'importati...,ANC,2001-06-27,<NA>,2502738.54,1117004.57
3,48,DD 14989/1,DD,--,Demande définitive,14989,1,-,NaT,non renseigné,Non renseigné (valeur par défaut à l'importati...,ANC,2001-06-27,<NA>,2502738.33,1116951.27
4,49,DD 25882/1,DD,--,Demande définitive,25882,1,-,NaT,non renseigné,Non renseigné (valeur par défaut à l'importati...,ANC,2001-06-27,<NA>,2502739.63,1116938.97


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 197224 entries, 0 to 197223
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   OBJECTID           197224 non-null  string        
 1   ID_DOSSIER         197224 non-null  string        
 2   TYPE_DOSSIER       197224 non-null  string        
 3   TYPE_OPERATION     197189 non-null  string        
 4   NOM_DOSSIER        197215 non-null  string        
 5   NO_DOSSIER         197224 non-null  string        
 6   NO_COMPLEMENTAIRE  197223 non-null  string        
 7   STATUT             197217 non-null  string        
 8   DATE_DEPOT         128809 non-null  datetime64[us]
 9   DESCRIPTION        197217 non-null  string        
 10  OPERATION          197221 non-null  string        
 11  ORIGINE            197221 non-null  string        
 12  DATE_MAJ_2         197188 non-null  datetime64[us]
 13  LIEN_SAD           128813 non-null  string        
 14 

## Missing data
'head()' shows placeholder values where data is absent. Checking how common they are, sence 'info()' only counts true nulls.

In [5]:
for col in df.select_dtypes("string").columns:
    print(col, df[col].value_counts().head(3).to_dict())

OBJECTID {'45': 1, '46': 1, '47': 1}
ID_DOSSIER {'DD 1568': 3, 'DD 104289/1': 3, 'DD 93616/1': 2}
TYPE_DOSSIER {'DD': 103662, 'APA': 79376, 'APAT': 6037}
TYPE_OPERATION {'--': 147397, 'REN': 17361, 'PI': 14815}
NOM_DOSSIER {'Demande définitive': 103660, 'Autorisation procédure accélérée': 79367, 'Autorisation par annonce de travaux': 6037}
NO_DOSSIER {'93968': 12, '92130': 12, '23014': 9}
NO_COMPLEMENTAIRE {'1': 184202, '2': 9417, '3': 2516}
STATUT {'-': 68411, 'TERMINE': 62923, 'ARCHIVE': 16943}
DESCRIPTION {'non renseigné': 68411, "rénovation d'un appartement au 2ème étage": 2148, "rénovation d'un appartement au 1er étage": 1979}
OPERATION {"Non renseigné (valeur par défaut à l'importation des données)": 147429, "Travaux d'entretien et de rénovation intérieure ou extérieure, sans création de surfaces supplémentaires destinées à l'habitat ou aux activités et sans changement d'affectation important": 17361, "Construction neuve de peu d'importance, isolée ou attenante au bâtiment princi

The top values only reveal frequent placeholders. The next three checks look for rarer ones:
every value of the low-cardinality columns, blank strings in all text columns, and all short descriptions.

In [6]:
small = [
    "TYPE_DOSSIER",
    "TYPE_OPERATION",
    "NOM_DOSSIER",
    "NO_COMPLEMENTAIRE",
    "STATUT",
    "OPERATION",
    "ORIGINE",
]

for col in small:
    print(col, df[col].nunique(), sorted(df[col].dropna().unique()))

TYPE_DOSSIER 11 ['APA', 'APAT', 'APL', 'DD', 'DP', 'DR', 'LER', 'LER-PA', 'LER-PAT', 'M', 'MPA']
TYPE_OPERATION 10 ['--', 'AFF', 'AGR', 'AGV', 'AM', 'EQU', 'NOU', 'PI', 'REN', 'VIL']
NOM_DOSSIER 12 ['Autorisation par annonce de travaux', 'Autorisation par procédure accélérée', 'Autorisation procédure accélérée', 'Demande de renseignements', 'Demande définitive', 'Demande préalable', 'Démolition', 'Démolition procédure accéléré', 'Procédure accélérée', 'Requête en approbation sur la loi des eaux et des routes', 'Requête en approbation sur la loi des eaux et des routes par annonce de travaux', 'Requête en approbation sur la loi des eaux et des routes par procédure accélérée']
NO_COMPLEMENTAIRE 18 ['0', '1', '10', '11', '12', '13', '14', '15', '16', '2', '3', '31', '4', '5', '6', '7', '8', '9']
STATUT 21 ['-', 'ABANDONNE', 'ACCEPTE', 'ARCHANT', 'ARCHIVE', 'AUTORISATION ANNULEE', 'CADUQUE', 'CHANTIER', 'CLOTURE ADMIN', 'DECIDE', 'EN SUSPENS', 'ENREGISTRE', 'ENREGISTREMENT', 'INSTRUCTION', 

In [7]:
df.select_dtypes("string").apply(lambda s: s.str.strip().eq("").sum())

OBJECTID             0
ID_DOSSIER           0
TYPE_DOSSIER         0
TYPE_OPERATION       0
NOM_DOSSIER          0
NO_DOSSIER           0
NO_COMPLEMENTAIRE    0
STATUT               0
DESCRIPTION          0
OPERATION            0
ORIGINE              0
LIEN_SAD             0
E                    0
N                    0
dtype: int64

In [8]:
d = df["DESCRIPTION"].dropna()
d[d.str.len() <= 15].value_counts()
print(d[d.str.len() <= 15].value_counts().to_string())

DESCRIPTION
non renseigné      68411
piscine              803
véranda              727
jardin d'hiver       315
jour en toiture      137
garage               133
Piscine              115
villa - garage       113
Véranda               97
villa                 79
couvert               68
abri de jardin        64
clôture               41
villa et garage       36
villa - parking       30
mur anti-bruit        29
cabane à outils       26
véranda -             24
hangar agricole       23
sas d'entrée          22
pergola               22
piscine -             20
mur                   17
portail               16
mur anti bruit        16
Garage                14
mur de clôture        13
Véranda -             11
garage -              10
villa -                9
serre de jardin        9
villa - couvert        9
abri à vélos           9
marquise               8
dragage du port        8
couvert à vélos        8
mur antibruit          8
couvert à bois         7
couvert -              6
serre        


### Placeholders found

The export uses exactly four placeholder values: `--` in `TYPE_OPERATION`, `-` in `STATUT`,
`non renseigné` in `DESCRIPTION` and `Non renseigné (valeur par défaut à l'importation des données)`
in `OPERATION`.

### Nulls vs placeholders

Counting both per column with the complete list.

In [9]:
import pandas as pd

placeholders = [
    "--",
    "-",
    "non renseigné",
    "Non renseigné (valeur par défaut à l'importation des données)",
]

pd.DataFrame({"nulls": df.isna().sum(), "placeholders": df.isin(placeholders).sum()})

,nulls,placeholders
OBJECTID,0,0
ID_DOSSIER,0,0
TYPE_DOSSIER,0,0
TYPE_OPERATION,35,147397
NOM_DOSSIER,9,0
NO_DOSSIER,0,0
NO_COMPLEMENTAIRE,1,0
STATUT,7,68411
DATE_DEPOT,68415,0
DESCRIPTION,7,68411


### Are the missing fields the same records?

Checking whether the records without a description are also the ones without a date, status and link.

In [10]:
no_text = df["DESCRIPTION"].isna() | df["DESCRIPTION"].isin(placeholders)

pd.DataFrame(
    {
        "no date": df.loc[no_text, "DATE_DEPOT"].isna().sum(),
        "no status": df.loc[no_text, "STATUT"].isin(placeholders).sum(),
        "no link": df.loc[no_text, "LIEN_SAD"].isna().sum(),
        "origin ANC": df.loc[no_text, "ORIGINE"].eq("ANC").sum(),
    },
    index=[f"records without text: {no_text.sum()}"],
)

,no date,no status,no link,origin ANC
records without text: 68418,68412,68411,68411,68408


## Labels

Setting the empty records aside and counting the operation codes among the records that do have a description.

In [11]:
with_text = df[~no_text]

with_text["TYPE_OPERATION"].value_counts(dropna=False)

TYPE_OPERATION
--     79022
REN    17361
PI     14815
EQU     4549
VIL     4139
NOU     4011
AGR     2324
AGV     1992
AFF      555
AM        38
Name: count, dtype: int64[pyarrow]

## Labels over time

Counting labelled and unlabelled records per filing year.

In [12]:
year = with_text["DATE_DEPOT"].dt.year
labelled = with_text["TYPE_OPERATION"].ne("--")

pd.crosstab(year, labelled)

TYPE_OPERATION,False,True
DATE_DEPOT,,
1995.0,1,0
1996.0,3224,11
1997.0,2716,8
1998.0,2884,19
1999.0,952,2099
2000.0,665,2380
2001.0,657,2431
2002.0,561,2469
2003.0,533,2624


## Class mix over time

Share of each operation code among labelled records, using the four calendar periods
from the research plan: up to 2010, 2011–2012, 2013 and 2014–2026.

In [13]:
labelled_rows = with_text[labelled]
period = pd.cut(
    labelled_rows["DATE_DEPOT"].dt.year,
    bins=[0, 2010, 2012, 2013, 2026],
    labels=["≤2010", "2011–2012", "2013", "2014–2026"],
)

pd.crosstab(labelled_rows["TYPE_OPERATION"], period, normalize="columns").round(3)

DATE_DEPOT,≤2010,2011–2012,2013,2014–2026
TYPE_OPERATION,,,,
AFF,0.014,0.008,0.008,0.005
AGR,0.043,0.038,0.038,0.063
AGV,0.041,0.032,0.030,0.044
AM,0.001,0.002,0.000,0.000
EQU,0.091,0.100,0.080,0.092
NOU,0.058,0.056,0.061,0.164
PI,0.320,0.302,0.389,0.209
REN,0.379,0.408,0.334,0.235
VIL,0.053,0.055,0.060,0.189
